# 🤟 A-Y Sign Language — YOLOv8 Local Pipeline
### (Excludes J and Z — handled separately by LSTM)

## Run order:
| Cell | What it does | Run when |
|------|-------------|----------|
| 1 | Install ultralytics + tools | Once |
| 2 | Config & paths | Every time |
| 3 | Collect images (webcam) | To gather raw images |
| 4 | Label images (LabelImg) | After collecting |
| 5 | Check labeling progress | Anytime |
| 6 | Create train/val split + dataset.yaml | After labeling some |
| 7 | Initial training (small, fast) | After Cell 6 |
| 8 | Auto-label remaining images | After Cell 7 |
| 9 | Final split (manual + auto labels) | After Cell 8 |
| 10 | Final training (better model) | After Cell 9 |
| 11 | Evaluate per-class results | After Cell 10 |
| 12 | Export final model | Last step |

---
# CELL 1 — Install Dependencies
### ▶️ Run ONCE

In [1]:
import subprocess, sys
from ultralytics import YOLO
import torch

In [24]:
!pip uninstall torch torchvision torchaudio -y
import pip
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

Found existing installation: torch 2.8.0
Uninstalling torch-2.8.0:
  Successfully uninstalled torch-2.8.0
Found existing installation: torchvision 0.23.0
Uninstalling torchvision-0.23.0:
  Successfully uninstalled torchvision-0.23.0


You can safely remove it manually.
You can safely remove it manually.


Looking in indexes: https://download.pytorch.org/whl/cu121
     ---------------------------------------- 0.0/2.4 GB ? eta -:--:--
     ---------------------------------------- 0.0/2.4 GB 16.8 MB/s eta 0:02:26
     ---------------------------------------- 0.0/2.4 GB 16.8 MB/s eta 0:02:26
     ---------------------------------------- 0.0/2.4 GB 11.5 MB/s eta 0:03:32
     ---------------------------------------- 0.0/2.4 GB 12.9 MB/s eta 0:03:10
     ---------------------------------------- 0.0/2.4 GB 12.7 MB/s eta 0:03:12
     ---------------------------------------- 0.0/2.4 GB 12.1 MB/s eta 0:03:21
     ---------------------------------------- 0.0/2.4 GB 12.1 MB/s eta 0:03:21
     ---------------------------------------- 0.0/2.4 GB 12.1 MB/s eta 0:03:22
     ---------------------------------------- 0.0/2.4 GB 12.0 MB/s eta 0:03:22
     ---------------------------------------- 0.0/2.4 GB 12.0 MB/s eta 0:03:22
     ---------------------------------------- 0.0/2.4 GB 12.0 MB/s eta 0:03:22
 

In [2]:
packages = ['ultralytics', 'labelImg', 'seaborn', 'pyyaml']
for pkg in packages:
    print(f'Installing {pkg}...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])


print('\n✅ Installed!')
print(f'   torch  : {torch.__version__}')
print(f'   CUDA   : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'   GPU    : {torch.cuda.get_device_name(0)}')

Installing ultralytics...
Installing labelImg...
Installing seaborn...
Installing pyyaml...

✅ Installed!
   torch  : 2.5.1+cu121
   CUDA   : True
   GPU    : NVIDIA GeForce GTX 1650


---
# CELL 2 — Config & Setup
### ▶️ Run EVERY TIME

In [3]:
import os, shutil, random, time
import yaml

# ⬇️ YOUR PROJECT PATH
PROJECT_PATH = r'E:\Study\Projects\REAL-TIME OBJECT DETECTION FOR ASSISTING COMMUNICATION IN NON-VERBAL INDIVIDUALS'
# ── Paths ──────────────────────────────────────────
DATA_PATH    = os.path.join(PROJECT_PATH, 'AY_data')
RAW_PATH     = os.path.join(PROJECT_PATH, 'Tensorflow', 'workspace', 'images', 'collectedimages')          # collected images + labels live here per class
DATASET_PATH = os.path.join(DATA_PATH, 'dataset')      # final YOLO train/val split
MODELS_PATH  = os.path.join(DATA_PATH, 'models')

# ── Classes (A-Y excluding J and Z) ───────────────
# CLASSES = [c for c in 'MN' if c not in ('J', 'Z')] 
CLASSES = [c for c in 'ABCDEFGHIJKLMNOPQRSTUVWXYZ' if c not in ('J', 'Z')]
print(f'Classes ({len(CLASSES)}): {CLASSES}')

for c in CLASSES:
    os.makedirs(os.path.join(RAW_PATH, c), exist_ok=True)
os.makedirs(MODELS_PATH, exist_ok=True)

# ── Collection settings ───────────────────────────
IMAGES_PER_CLASS = 20   # target images per class

# ── Training settings ─────────────────────────────
INITIAL_MODEL = 'yolov8n.pt'   # nano — fast initial pass for auto-labeling
# FINAL_MODEL   = 'yolov8s.pt'   # small — better final accuracy
FINAL_MODEL = 'sign_lang_best.pt'

# ── classes.txt for LabelImg (YOLO format) ────────
classes_txt = os.path.join(DATA_PATH, 'classes.txt')
with open(classes_txt, 'w') as f:
    f.write('\n'.join(CLASSES))

print(f'\n✅ Setup complete!')
print(f'   Raw images path : {RAW_PATH}')
print(f'   Dataset path    : {DATASET_PATH}')
print(f'   Target images   : {IMAGES_PER_CLASS} per class')
print(f'   classes.txt     : {classes_txt}')

Classes (24): ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y']

✅ Setup complete!
   Raw images path : E:\Study\Projects\REAL-TIME OBJECT DETECTION FOR ASSISTING COMMUNICATION IN NON-VERBAL INDIVIDUALS\Tensorflow\workspace\images\collectedimages
   Dataset path    : E:\Study\Projects\REAL-TIME OBJECT DETECTION FOR ASSISTING COMMUNICATION IN NON-VERBAL INDIVIDUALS\AY_data\dataset
   Target images   : 20 per class
   classes.txt     : E:\Study\Projects\REAL-TIME OBJECT DETECTION FOR ASSISTING COMMUNICATION IN NON-VERBAL INDIVIDUALS\AY_data\classes.txt


In [4]:
# ── Cell 2b — Make LabelImg default to YOLO format permanently ──
labelimg_data = os.path.join(PROJECT_PATH, 'Tensorflow', 'labelimg', 'data')

with open(os.path.join(labelimg_data, 'predefined_classes.txt'), 'w') as f:
    f.write('\n'.join(CLASSES))

print('Done! LabelImg will now default to YOLO format permanently.')
print('No more Ctrl+Y needed.')

Done! LabelImg will now default to YOLO format permanently.
No more Ctrl+Y needed.


---
# CELL 3 — Collect Images (Webcam)
### ▶️ Run to write the collection script, then run it in a SEPARATE terminal
### Controls: SPACE = capture | N = next class | Q = quit
### Tip: vary hand angle, distance, lighting for each shot

In [7]:
collect_script = f'''
import cv2, os, time

RAW_PATH = r"{RAW_PATH}"
CLASSES  = {CLASSES}
TARGET   = {IMAGES_PER_CLASS}

cap = cv2.VideoCapture(0)
if not cap.isOpened():
    cap = cv2.VideoCapture(1)

print("Controls: SPACE=capture | N=next class | Q=quit")

class_idx = 0
while class_idx < len(CLASSES):
    current_class = CLASSES[class_idx]
    class_dir = os.path.join(RAW_PATH, current_class)
    existing = len([f for f in os.listdir(class_dir) if f.endswith(".jpg")])
    count = existing

    while count < TARGET:
        ret, frame = cap.read()
        if not ret: break
        frame = cv2.flip(frame, 1)
        display = frame.copy()
        cv2.rectangle(display, (0,0), (display.shape[1], 90), (0,0,0), -1)
        cv2.putText(display, f"Class: {{current_class}}  ({{count}}/{{TARGET}})",
                   (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0,255,0), 2)
        cv2.putText(display, "SPACE=capture | N=next | Q=quit",
                   (20, 75), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200,200,200), 1)
        cv2.imshow("Collecting A-Y Images", display)

        key = cv2.waitKey(1) & 0xFF
        if key == ord(" "):
            fname = f"{{current_class}}_{{int(time.time()*1000)}}.jpg"
            cv2.imwrite(os.path.join(class_dir, fname), frame)
            count += 1
            print(f"  {{current_class}}: {{count}}/{{TARGET}} saved")
        elif key == ord("n"):
            break
        elif key == ord("q"):
            cap.release(); cv2.destroyAllWindows(); exit()

    class_idx += 1

cap.release()
cv2.destroyAllWindows()
print("\\n✅ Collection done!")
'''

script_path = os.path.join(PROJECT_PATH, 'collect_ay.py')
with open(script_path, 'w', encoding='utf-8') as f:
    f.write(collect_script)

print(f'✅ Script saved: {script_path}')
print()
print('Run in a SEPARATE terminal:')
print(f'   cd "{PROJECT_PATH}"')
print( '   ObjEnv\\Scripts\\activate')
print( '   python collect_ay.py')

✅ Script saved: E:\Study\Projects\REAL-TIME OBJECT DETECTION FOR ASSISTING COMMUNICATION IN NON-VERBAL INDIVIDUALS\collect_ay.py

Run in a SEPARATE terminal:
   cd "E:\Study\Projects\REAL-TIME OBJECT DETECTION FOR ASSISTING COMMUNICATION IN NON-VERBAL INDIVIDUALS"
   ObjEnv\Scripts\activate
   python collect_ay.py


---
# CELL 4 — Label Images (LabelImg)
### ▶️ Launches LabelImg directly
### ⚠️ IMPORTANT: In LabelImg, click the format button until it says "YOLO"
### Saves .txt label files next to each .jpg automatically
### Shortcut: W = draw box, A/D = prev/next image, Ctrl+S = save

In [14]:
LABEL_CLASS    = 'M'   # ← change to N when done with M
LABELIMG_PATH  = os.path.join(PROJECT_PATH, 'Tensorflow', 'labelimg')
class_dir      = os.path.join(RAW_PATH, LABEL_CLASS)

print(f'Class    : {LABEL_CLASS}')
print(f'Folder   : {class_dir}')
print(f'Images   : {len([f for f in os.listdir(class_dir) if f.endswith(".jpg")])}')
print(f'Exists   : {os.path.exists(class_dir)}')

import subprocess
subprocess.Popen([
    sys.executable,
    os.path.join(LABELIMG_PATH, 'labelImg.py'),
    class_dir,
    os.path.join(DATA_PATH, 'classes.txt'),
    class_dir
])
print('✅ LabelImg launched!')

Class    : M
Folder   : E:\Study\Projects\REAL-TIME OBJECT DETECTION FOR ASSISTING COMMUNICATION IN NON-VERBAL INDIVIDUALS\Tensorflow\workspace\images\collectedimages\M
Images   : 53
Exists   : True
✅ LabelImg launched!


---
# CELL 5 — Check Labeling Progress
### Run anytime to see how much is labeled per class

### ── Convert existing VOC XML labels → YOLO txt ───

In [17]:

import xml.etree.ElementTree as ET

def voc_xml_to_yolo_txt(xml_path, classes):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    size = root.find('size')
    W    = float(size.find('width').text)
    H    = float(size.find('height').text)
    lines = []
    for obj in root.findall('object'):
        name = obj.find('name').text
        if name not in classes: continue
        cls_id = classes.index(name)
        bb   = obj.find('bndbox')
        xmin = float(bb.find('xmin').text)
        ymin = float(bb.find('ymin').text)
        xmax = float(bb.find('xmax').text)
        ymax = float(bb.find('ymax').text)
        cx = ((xmin + xmax) / 2) / W
        cy = ((ymin + ymax) / 2) / H
        w  = (xmax - xmin) / W
        h  = (ymax - ymin) / H
        lines.append(f'{cls_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}')
    return lines

converted = skipped = 0
for c in CLASSES:
    cdir = os.path.join(RAW_PATH, c)
    if not os.path.exists(cdir): continue
    for f in os.listdir(cdir):
        if not f.endswith('.xml'): continue
        xml_path = os.path.join(cdir, f)
        txt_path = xml_path.replace('.xml', '.txt')
        if os.path.exists(txt_path):
            skipped += 1; continue
        lines = voc_xml_to_yolo_txt(xml_path, CLASSES)
        if lines:
            with open(txt_path, 'w') as out:
                out.write('\n'.join(lines))
            converted += 1

print(f'Converted : {converted} XML → TXT')
print(f'Skipped   : {skipped} (already had .txt)')

Converted : 1231 XML → TXT
Skipped   : 0 (already had .txt)


In [9]:
print('📊 Labeling Progress')
print('='*50)
print(f'{"Class":<8}{"Images":<10}{"Labeled":<10}{"Unlabeled":<10}')
print('-'*50)

total_images, total_labeled = 0, 0
for c in CLASSES:
    cdir = os.path.join(RAW_PATH, c)
    imgs = [f for f in os.listdir(cdir) if f.endswith('.jpg')]
    labeled = [f for f in imgs if os.path.exists(
        os.path.join(cdir, f.replace('.jpg', '.txt')))]
    total_images  += len(imgs)
    total_labeled += len(labeled)
    print(f'{c:<8}{len(imgs):<10}{len(labeled):<10}{len(imgs)-len(labeled):<10}')

print('-'*50)
print(f'{"TOTAL":<8}{total_images:<10}{total_labeled:<10}{total_images-total_labeled:<10}')
print('='*50)

MIN_LABELED = 15
ready = [c for c in CLASSES if len([
    f for f in os.listdir(os.path.join(RAW_PATH,c)) if f.endswith('.jpg')
    and os.path.exists(os.path.join(RAW_PATH,c,f.replace('.jpg','.txt')))
]) >= MIN_LABELED]

print(f'\nClasses with ≥{MIN_LABELED} labeled images: {len(ready)}/{len(CLASSES)}')
if len(ready) < len(CLASSES):
    missing = [c for c in CLASSES if c not in ready]
    print(f'Still need labeling: {missing}')
else:
    print('✅ Ready for Cell 6!')

📊 Labeling Progress
Class   Images    Labeled   Unlabeled 
--------------------------------------------------
A       51        51        0         
B       53        53        0         
C       50        50        0         
D       52        52        0         
E       52        52        0         
F       51        51        0         
G       51        51        0         
H       42        42        0         
I       50        50        0         
K       51        51        0         
L       52        52        0         
M       73        73        0         
N       72        72        0         
O       51        51        0         
P       52        52        0         
Q       51        51        0         
R       52        52        0         
S       53        53        0         
T       50        50        0         
U       50        50        0         
V       50        50        0         
W       52        52        0         
X       58        58        0   

---
# CELL 6 — Create Train/Val Split + dataset.yaml
### ▶️ Run after labeling at least ~15 images per class
### Splits 80/20 and writes dataset.yaml for YOLO

In [10]:
def rebuild_split(source_dirs_per_class, dataset_path, classes, val_ratio=0.2, seed=42):
    """source_dirs_per_class: dict {class_name: [list of (img_path, label_path)]}"""
    random.seed(seed)
    for split in ['train', 'val']:
        os.makedirs(os.path.join(dataset_path, 'images', split), exist_ok=True)
        os.makedirs(os.path.join(dataset_path, 'labels', split), exist_ok=True)

    counts = {'train': 0, 'val': 0}
    for cls, pairs in source_dirs_per_class.items():
        random.shuffle(pairs)
        n_val = max(1, int(len(pairs) * val_ratio))
        val_pairs   = pairs[:n_val]
        train_pairs = pairs[n_val:]

        for split, split_pairs in [('train', train_pairs), ('val', val_pairs)]:
            for img_path, lbl_path in split_pairs:
                img_name = os.path.basename(img_path)
                lbl_name = os.path.basename(lbl_path)
                shutil.copy(img_path, os.path.join(dataset_path, 'images', split, img_name))
                shutil.copy(lbl_path, os.path.join(dataset_path, 'labels', split, lbl_name))
                counts[split] += 1
    return counts


# ── Gather labeled pairs per class ────────────────
labeled_pairs = {}
for c in CLASSES:
    cdir = os.path.join(RAW_PATH, c)
    pairs = []
    for f in os.listdir(cdir):
        if f.endswith('.jpg'):
            lbl = f.replace('.jpg', '.txt')
            lbl_path = os.path.join(cdir, lbl)
            if os.path.exists(lbl_path):
                pairs.append((os.path.join(cdir, f), lbl_path))
    labeled_pairs[c] = pairs
    print(f'  {c}: {len(pairs)} labeled images')

# ── Clean previous dataset and rebuild ────────────
if os.path.exists(DATASET_PATH):
    shutil.rmtree(DATASET_PATH)

counts = rebuild_split(labeled_pairs, DATASET_PATH, CLASSES)
print(f'\n✅ Split done: {counts}')

# ── Write dataset.yaml ─────────────────────────────
dataset_yaml = {
    'path': DATASET_PATH,
    'train': 'images/train',
    'val':   'images/val',
    'nc':    len(CLASSES),
    'names': CLASSES
}
yaml_path = os.path.join(DATA_PATH, 'dataset.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(dataset_yaml, f, default_flow_style=False)

print(f'✅ dataset.yaml written: {yaml_path}')
print(f'\n{yaml.dump(dataset_yaml, default_flow_style=False)}')

  A: 51 labeled images
  B: 53 labeled images
  C: 50 labeled images
  D: 52 labeled images
  E: 52 labeled images
  F: 51 labeled images
  G: 51 labeled images
  H: 42 labeled images
  I: 50 labeled images
  K: 51 labeled images
  L: 52 labeled images
  M: 73 labeled images
  N: 72 labeled images
  O: 51 labeled images
  P: 52 labeled images
  Q: 51 labeled images
  R: 52 labeled images
  S: 53 labeled images
  T: 50 labeled images
  U: 50 labeled images
  V: 50 labeled images
  W: 52 labeled images
  X: 58 labeled images
  Y: 52 labeled images

✅ Split done: {'train': 1024, 'val': 247}
✅ dataset.yaml written: E:\Study\Projects\REAL-TIME OBJECT DETECTION FOR ASSISTING COMMUNICATION IN NON-VERBAL INDIVIDUALS\AY_data\dataset.yaml

names:
- A
- B
- C
- D
- E
- F
- G
- H
- I
- K
- L
- M
- N
- O
- P
- Q
- R
- S
- T
- U
- V
- W
- X
- Y
nc: 24
path: E:\Study\Projects\REAL-TIME OBJECT DETECTION FOR ASSISTING COMMUNICATION IN
  NON-VERBAL INDIVIDUALS\AY_data\dataset
train: images/train
val: im

---
# CELL 7 — Initial Training (Fast Pass)
### ▶️ Quick yolov8n training — used ONLY to auto-label remaining images
### Takes ~10-20 min on GTX 1650

In [ ]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()

model_initial = YOLO(FINAL_MODEL)  # loading the sign_lang_best.pt

results = model_initial.train(
    data    = yaml_path,
    epochs  = 50,
    imgsz   = 640,
    batch   = 4,
    device  = 0,
    project = MODELS_PATH,
    name    = 'final_v2',
    exist_ok= True,
    verbose = True
)

initial_weights = os.path.join(MODELS_PATH, 'final_v2', 'weights', 'best.pt')
print(f'\n✅ Initial model saved: {initial_weights}')

---
# CELL 8 — Auto-Label Remaining Images
### ▶️ Uses initial model to label unlabeled images
### High-confidence labels accepted automatically
### Low-confidence flagged for manual review

In [ ]:
import json

AUTO_LABEL_CONF = 0.5   # accept auto-labels above this confidence

auto_model = YOLO(initial_weights)

auto_labeled, flagged_review = 0, []

for c in CLASSES:
    cdir = os.path.join(RAW_PATH, c)
    cls_idx = CLASSES.index(c)

    for f in os.listdir(cdir):
        if not f.endswith('.jpg'): continue
        lbl_path = os.path.join(cdir, f.replace('.jpg', '.txt'))
        if os.path.exists(lbl_path):
            continue  # already labeled (manually)

        img_path = os.path.join(cdir, f)
        results  = auto_model(img_path, conf=AUTO_LABEL_CONF, verbose=False)

        if results[0].boxes and len(results[0].boxes) > 0:
            # Use highest-confidence box, force-assign to this folder's class
            box  = results[0].boxes[0]
            xywhn = box.xywhn[0].tolist()  # normalized x_center,y_center,w,h
            conf  = float(box.conf[0])

            with open(lbl_path, 'w') as lf:
                lf.write(f'{cls_idx} {xywhn[0]:.6f} {xywhn[1]:.6f} {xywhn[2]:.6f} {xywhn[3]:.6f}\n')

            auto_labeled += 1
            if conf < 0.7:
                flagged_review.append({'class': c, 'file': f, 'confidence': round(conf,2)})
        else:
            flagged_review.append({'class': c, 'file': f, 'confidence': 0})

# Save review list
review_path = os.path.join(DATA_PATH, 'manual_review.json')
with open(review_path, 'w') as f:
    json.dump(flagged_review, f, indent=2)

print(f'✅ Auto-labeled: {auto_labeled} images')
print(f'⚠️  Flagged for review (low/no confidence): {len(flagged_review)}')
print(f'   Saved to: {review_path}')
print()
print('Review low-confidence labels by re-running Cell 4 with')
print('LABEL_CLASS set to the flagged classes, and fix/confirm boxes.')

---
# CELL 9 — Final Split (Manual + Auto Labels)
### ▶️ Rebuilds dataset.yaml split using ALL labeled images now

In [ ]:
# ── Gather ALL labeled pairs (manual + auto) ──────
labeled_pairs = {}
for c in CLASSES:
    cdir = os.path.join(RAW_PATH, c)
    pairs = []
    for f in os.listdir(cdir):
        if f.endswith('.jpg'):
            lbl = f.replace('.jpg', '.txt')
            lbl_path = os.path.join(cdir, lbl)
            if os.path.exists(lbl_path):
                pairs.append((os.path.join(cdir, f), lbl_path))
    labeled_pairs[c] = pairs
    print(f'  {c}: {len(pairs)} total labeled images')

if os.path.exists(DATASET_PATH):
    shutil.rmtree(DATASET_PATH)

counts = rebuild_split(labeled_pairs, DATASET_PATH, CLASSES)
print(f'\n✅ Final split: {counts}')

with open(yaml_path, 'w') as f:
    yaml.dump(dataset_yaml, f, default_flow_style=False)
print(f'✅ dataset.yaml refreshed')

---
# CELL 10 — Final Training
### ▶️ Better model (yolov8s), more epochs, on full dataset
### Takes ~1-2 hours on GTX 1650 — go grab a coffee ☕

In [5]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

2.5.1+cu121
True
1
NVIDIA GeForce GTX 1650


In [11]:
print(yaml_path)
print(FINAL_MODEL)
print(MODELS_PATH)

E:\Study\Projects\REAL-TIME OBJECT DETECTION FOR ASSISTING COMMUNICATION IN NON-VERBAL INDIVIDUALS\AY_data\dataset.yaml
sign_lang_best.pt
E:\Study\Projects\REAL-TIME OBJECT DETECTION FOR ASSISTING COMMUNICATION IN NON-VERBAL INDIVIDUALS\AY_data\models


In [ ]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()

# model_final = YOLO(FINAL_MODEL)
model_final = YOLO(FINAL_MODEL)

results = model_final.train(
    data     = yaml_path,
    epochs   = 50,
    imgsz    = 640,
    batch    = 4,
    patience = 15,         # early stopping
    device   = 0,
    project  = MODELS_PATH,
    name     = 'final_v2',
    exist_ok = True,
    verbose  = True
)

final_weights = os.path.join(MODELS_PATH, 'final_v2', 'weights', 'final_v2.pt')
print(f'\n✅ Final model saved: {final_weights}')

New https://pypi.org/project/ultralytics/8.4.76 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.64  Python-3.9.13 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=E:\Study\Projects\REAL-TIME OBJECT DETECTION FOR ASSISTING COMMUNICATION IN NON-VERBAL INDIVIDUALS\AY_data\dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=30

---
# CELL 11 — Evaluate Per-Class Results

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from PIL import Image as PILImage
import io

eval_model = YOLO(final_weights)
metrics    = eval_model.val(data=yaml_path)

print('\n📊 PER-CLASS RESULTS')
print('='*50)
print(f'{"Class":<8}{"mAP50":<10}{"Precision":<12}{"Recall":<10}')
print('-'*50)

results_summary = {}
for i, c in enumerate(CLASSES):
    map50 = metrics.box.ap50[i] if i < len(metrics.box.ap50) else 0
    p     = metrics.box.p[i]    if i < len(metrics.box.p)    else 0
    r     = metrics.box.r[i]    if i < len(metrics.box.r)    else 0
    results_summary[c] = map50
    grade = '✅' if map50>=0.8 else '⚠️ ' if map50>=0.5 else '❌'
    print(f'{grade} {c:<6}{map50:<10.2f}{p:<12.2f}{r:<10.2f}')

print('-'*50)
print(f'Overall mAP50: {metrics.box.map50:.2f}')
print('='*50)

# Confusion matrix
cm_path = os.path.join(MODELS_PATH, 'final', 'confusion_matrix.png')
if os.path.exists(cm_path):
    display(PILImage.open(cm_path))

weak = [c for c,v in results_summary.items() if v < 0.7]
if weak:
    print(f'\n⚠️  Weak classes (mAP50 < 0.7): {weak}')
    print('   Consider collecting more images for these in Cell 3')
else:
    print('\n✅ All classes look good!')

---
# CELL 12 — Export Final Model

In [ ]:
import shutil

# Copy best model to project root with clear name
output_path = os.path.join(PROJECT_PATH, 'sign_lang_AY_best.pt')
shutil.copy(final_weights, output_path)
print(f'✅ Model copied to: {output_path}')

# Export to additional formats
export_model = YOLO(final_weights)

print('\nExporting to ONNX...')
export_model.export(format='onnx')

print('Exporting to TFLite (for mobile)...')
export_model.export(format='tflite')

print('\n✅ All exports done!')
print(f'   PyTorch : {output_path}')
print(f'   ONNX/TFLite saved next to: {final_weights}')
print()
print('Use sign_lang_AY_best.pt in your detect_combined.py script')